# Non-Graphical Exploratory Data Analysis (EDA)

Non-graphical EDA uses statistical measures to summarize and understand data distributions and relationships **without** visual representations.

## Overview
| | Methods |
|---|---|
| **Univariate** | mean, median, mode, quantiles, variance, std, skewness, kurtosis |
| **Multivariate** | contingency tables, covariance matrix, correlation matrix |

In [1]:
# Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load apartment data
df = pd.read_csv('../data/apartments_data_enriched_cleaned.csv', )
print('Shape:', df.shape)
df.head(3)

Shape: (774, 17)


,Unnamed: 0,web-scraper-order,address_raw,lat,lon,bfs_number,bfs_name,rooms,area,luxurious,price,price_per_m2,pop,pop_dens,emp,frg_pct,mean_taxable_income
0,0,1693998201-1,"Neuhusstrasse 6, 8630 Rüti ZH, ZH",47.252171,8.845797,118,Rüti (ZH),3.0,49.0,0,1441.0,29.41,12286,1221.272366,5053.0,24.841283,65362.042683
1,1,1693998233-172,"Widacherstrasse 5, 8630 Rüti ZH, ZH",47.252087,8.854919,118,Rüti (ZH),3.0,111.0,0,2600.0,23.42,12286,1221.272366,5053.0,24.841283,65362.042683
2,2,1693998256-331,"Widenweg 14, 8630 Rüti ZH, ZH",47.253670,8.853993,118,Rüti (ZH),3.0,58.0,0,1490.0,25.69,12286,1221.272366,5053.0,24.841283,65362.042683


---
## Univariate Non-Graphical EDA

### Measures of Central Tendency: Mean, Median, Mode

In [2]:
# Mean, median, mode for apartment price
print(f"Mean:   {df['price'].mean():.2f}")
print(f"Median: {df['price'].median():.2f}")
print(f"Mode:   {df['price'].mode()[0]:.2f}")
print()
# When mean > median: right-skewed distribution
print('Note: mean > median indicates right-skewed distribution (long tail to the right)')

Mean:   2652.96
Median: 2390.00
Mode:   1750.00

Note: mean > median indicates right-skewed distribution (long tail to the right)


### Quantiles

Quantiles divide the distribution into equal-sized groups. Common quantiles:
- **Quartiles**: 4 groups (Q1=25%, Q2=50%=median, Q3=75%)
- **Deciles**: 10 groups
- **Percentiles**: 100 groups

In [3]:
# Quantiles for selected numerical variables
num_cols = ['price', 'rooms', 'area']
df[num_cols].quantile([0.05, 0.25, 0.50, 0.75, 0.95]).round(2)

,price,rooms,area
0.05,1356.50,1.0,26.00
0.25,1849.25,2.5,63.25
0.50,2390.00,3.5,86.00
0.75,3027.50,4.5,108.00
0.95,4970.50,5.5,160.00


### Measures of Dispersion: Variance and Standard Deviation

In [4]:
# Variance and standard deviation
print('Sample Variance:')
print(df[num_cols].var().round(2))
print()
print('Sample Standard Deviation:')
print(df[num_cols].std().round(2))

Sample Variance:
price    1643291.81
rooms          1.64
area        1560.96
dtype: float64

Sample Standard Deviation:
price    1281.91
rooms       1.28
area       39.51
dtype: float64


### Skewness

Skewness measures the asymmetry of a distribution:
- **Positive (right-skewed)**: long tail to the right, mean > median
- **Negative (left-skewed)**: long tail to the left, mean < median
- **Zero**: symmetric distribution

In [5]:
# Skewness
print('Skewness:')
print(df[num_cols].skew().round(3))
print()
# Compare mean vs median as rule of thumb
for col in num_cols:
    mean = df[col].mean()
    median = df[col].median()
    direction = 'right-skewed' if mean > median else 'left-skewed' if mean < median else 'symmetric'
    print(f'{col}: mean={mean:.2f}, median={median:.2f} → {direction}')

Skewness:
price    2.182
rooms    0.195
area     0.888
dtype: float64

price: mean=2652.96, median=2390.00 → right-skewed
rooms: mean=3.42, median=3.50 → left-skewed
area: mean=88.90, median=86.00 → right-skewed


### Kurtosis

Kurtosis measures the "tailedness" of a distribution — how heavy or light the tails are relative to a normal distribution.
- **Excess kurtosis > 0**: heavier tails than normal (leptokurtic)
- **Excess kurtosis < 0**: lighter tails than normal (platykurtic)
- **Excess kurtosis ≈ 0**: similar to normal (mesokurtic)

> Note: pandas returns **excess kurtosis** (Fisher's definition, where normal = 0).

In [6]:
# Kurtosis (excess kurtosis, pandas default)
print('Excess kurtosis:')
print(df[num_cols].kurtosis().round(3))

Excess kurtosis:
price    6.933
rooms    0.146
area     1.964
dtype: float64


### Summary Statistics: `describe()`

A quick way to get an overview of all the above measures at once.

In [7]:
# Full summary statistics
df[num_cols].describe().round(2)

,price,rooms,area
count,774.00,774.00,774.00
mean,2652.96,3.42,88.90
std,1281.91,1.28,39.51
min,86.00,1.00,8.00
25%,1849.25,2.50,63.25
50%,2390.00,3.50,86.00
75%,3027.50,4.50,108.00
max,9950.00,8.50,282.00


---
## Multivariate Non-Graphical EDA

### Contingency Table (Cross-Table)

A contingency table shows the frequency distribution of two categorical (or discretized) variables.

In [8]:
# Contingency table: rooms vs. luxurious category
contingency = pd.crosstab(
    df['rooms'].astype(int),
    df['luxurious'],
    margins=True,
    margins_name='Total'
)
contingency.index.name = 'rooms'
contingency.columns.name = 'luxurious'
contingency

luxurious,0,1,Total
rooms,,,
1,83,0,83
2,173,6,179
3,244,3,247
4,191,0,191
5,55,0,55
6,13,0,13
7,5,0,5
8,1,0,1
Total,765,9,774


### Covariance Matrix

Covariance measures the joint variability of two variables. It can be any value between −∞ and +∞. A positive value indicates that both variables tend to increase together; negative means one increases as the other decreases.

In [9]:
# Covariance matrix
df[num_cols].cov().round(2)

,price,rooms,area
price,1643291.81,773.83,30997.98
rooms,773.83,1.64,42.40
area,30997.98,42.40,1560.96


### Correlation Matrix

Correlation normalizes covariance to the range **[−1, +1]**, making it easier to interpret:
- **+1**: perfect positive linear relationship
- **−1**: perfect negative linear relationship
- **0**: no linear relationship

Common methods:
- **Pearson**: linear relationships (default)
- **Spearman**: rank-based, robust to outliers and non-linear monotone relationships

In [10]:
# Pearson correlation matrix
print('Pearson correlation:')
display(df.corr(method='pearson', numeric_only=True).round(3))

print('\nSpearman correlation:')
display(df.corr(method='spearman', numeric_only=True).round(3))

Pearson correlation:


,Unnamed: 0,lat,lon,bfs_number,rooms,area,luxurious,price,price_per_m2,pop,pop_dens,emp,frg_pct,mean_taxable_income
Unnamed: 0,1.000,0.080,0.208,-0.658,0.267,0.192,-0.017,-0.257,-0.518,-0.822,-0.765,-0.806,-0.385,0.021
lat,0.080,1.000,0.011,-0.086,0.078,0.075,-0.118,-0.147,-0.196,-0.143,-0.231,-0.166,-0.146,-0.304
lon,0.208,0.011,1.000,-0.020,0.124,0.085,0.006,-0.167,-0.311,-0.359,-0.457,-0.392,-0.537,-0.199
bfs_number,-0.658,-0.086,-0.020,1.000,-0.209,-0.165,0.022,0.184,0.390,0.703,0.665,0.676,0.315,-0.012
rooms,0.267,0.078,0.124,-0.209,1.000,0.837,-0.049,0.471,-0.474,-0.266,-0.314,-0.261,-0.272,0.021
area,0.192,0.075,0.085,-0.165,0.837,1.000,-0.019,0.612,-0.456,-0.218,-0.263,-0.215,-0.259,0.073
luxurious,-0.017,-0.118,0.006,0.022,-0.049,-0.019,1.000,0.134,0.129,0.021,0.045,0.023,0.002,0.106
price,-0.257,-0.147,-0.167,0.184,0.471,0.612,0.134,1.000,0.287,0.310,0.288,0.317,0.090,0.240
price_per_m2,-0.518,-0.196,-0.311,0.390,-0.474,-0.456,0.129,0.287,1.000,0.612,0.620,0.619,0.352,0.178
pop,-0.822,-0.143,-0.359,0.703,-0.266,-0.218,0.021,0.310,0.612,1.000,0.927,0.997,0.435,0.069



Spearman correlation:


,Unnamed: 0,lat,lon,bfs_number,rooms,area,luxurious,price,price_per_m2,pop,pop_dens,emp,frg_pct,mean_taxable_income
Unnamed: 0,1.000,0.056,0.207,-0.751,0.290,0.226,-0.017,-0.194,-0.539,-0.785,-0.723,-0.768,-0.437,-0.180
lat,0.056,1.000,0.007,-0.096,0.051,0.060,-0.117,-0.139,-0.256,-0.145,-0.196,-0.115,-0.092,-0.363
lon,0.207,0.007,1.000,-0.180,0.119,0.104,0.019,-0.159,-0.317,-0.181,-0.346,-0.193,-0.530,-0.308
bfs_number,-0.751,-0.096,-0.180,1.000,-0.247,-0.214,0.020,0.207,0.540,0.816,0.781,0.811,0.442,0.223
rooms,0.290,0.051,0.119,-0.247,1.000,0.862,-0.061,0.537,-0.499,-0.331,-0.335,-0.329,-0.281,-0.047
area,0.226,0.060,0.104,-0.214,0.862,1.000,-0.014,0.667,-0.500,-0.286,-0.285,-0.285,-0.251,-0.015
luxurious,-0.017,-0.117,0.019,0.020,-0.061,-0.014,1.000,0.087,0.106,0.027,0.070,0.020,0.009,0.099
price,-0.194,-0.139,-0.159,0.207,0.537,0.667,0.087,1.000,0.215,0.190,0.216,0.192,0.098,0.300
price_per_m2,-0.539,-0.256,-0.317,0.540,-0.499,-0.500,0.106,0.215,1.000,0.617,0.652,0.615,0.468,0.378
pop,-0.785,-0.145,-0.181,0.816,-0.331,-0.286,0.027,0.190,0.617,1.000,0.900,0.989,0.589,0.157


## Jupyter notebook --footer info--
(please always provide this at the end of each notebook)

In [11]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('------------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('------------------------------------')

------------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-01 07:37:29
Python Version: 3.13.5
------------------------------------
